# 07 — Classical NLP Modelling
## ShopEase Europe | Sentiment Analysis Project
**Objective:** Train and evaluate Naive Bayes and Logistic Regression 
baseline models for sentiment classification. Results establish a 
performance benchmark against which the XLM-RoBERTa transformer model 
will be measured in notebook 08.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from scipy.sparse import load_npz

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report, 
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## Load Preprocessed Data

In [2]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROCESSED_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed')
FIGURES_PATH = os.path.join(PROJECT_ROOT, 'reports', 'figures')
MODELS_PATH = os.path.join(PROJECT_ROOT, 'models')

# Load preprocessed dataframe and TF-IDF matrix from notebook 03
df = pd.read_csv(os.path.join(PROCESSED_PATH, 'preprocessed_reviews.csv'))
X = load_npz(os.path.join(PROCESSED_PATH, 'tfidf_matrix.npz'))

print(f"Dataframe loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"TF-IDF matrix loaded: {X.shape}")
print(f"Sentiment classes: {df['sentiment'].unique()}")

Dataframe loaded: 120,000 rows x 10 columns
TF-IDF matrix loaded: (120000, 1155)
Sentiment classes: ['positive' 'neutral' 'negative']


## Train-Test Split
Splitting the dataset into training and test sets, stratified by 
language to ensure all four primary languages are proportionally 
represented in both sets, which is critical given our multilingual dataset.

In [4]:
# First split: 80% train+val, 20% test
train_val_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    stratify=df['sentiment'],
    random_state=42
)

# Second split: split train_val into 87.5% train, 12.5% val
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.125,
    stratify=df.loc[train_val_idx, 'sentiment'],
    random_state=42
)

X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
y_train = df.loc[train_idx, 'sentiment']
y_val = df.loc[val_idx, 'sentiment']
y_test = df.loc[test_idx, 'sentiment']

print(f"Training set:   {X_train.shape[0]:,} rows ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]:,} rows ({X_val.shape[0]/len(df)*100:.1f}%)")
print(f"Test set:       {X_test.shape[0]:,} rows ({X_test.shape[0]/len(df)*100:.1f}%)")

print(f"\nTraining set sentiment distribution:")
print(y_train.value_counts(normalize=True).round(3) * 100)

print(f"\nLanguage distribution check across test set:")
print(df.loc[test_idx, 'language'].value_counts(normalize=True).round(3) * 100)

Training set:   84,000 rows (70.0%)
Validation set: 12,000 rows (10.0%)
Test set:       24,000 rows (20.0%)

Training set sentiment distribution:
sentiment
positive    68.0
negative    16.0
neutral     16.0
Name: proportion, dtype: float64

Language distribution check across test set:
language
en    44.0
fr    20.0
de    19.5
es    14.8
pt     0.9
nl     0.3
da     0.2
af     0.2
sv     0.1
it     0.0
Name: proportion, dtype: float64


### Stratification Note

Initial attempts to stratify by combined sentiment and language failed 
due to extremely rare language categories (Indonesian and Norwegian with 
only 1 record each) making stratification mathematically impossible. 
The split was adjusted to stratify by sentiment alone.

Verification confirms language distribution remained proportionally 
consistent across the test set without explicit stratification, a 
result of the large sample size (120,000 records) ensuring random 
sampling naturally preserves population proportions.